# Introduction to the PIC-SURE API
This is a tutorial notebook aimed to get the user quickly up and running with the PIC-SURE API. 

## PIC-SURE python API
### What is PIC-SURE?
As part of the *NHLBI BioData Catalyst® (BDC)* ecosystem, the Patient Information Commons: Standard Unification of Research Elements (PIC-SURE) platform has been integrating clinical and genomic datasets from multiple TOPMed and TOPMed-related studies funded by the National Heart, Lung, and Blood Institute (NHLBI). 

Original data exposed through the PIC-SURE API encompasses a large heterogeneity of data organization underneath. PIC-SURE hides this complexity and esposes the different study datasets in a single tabular format. By simplifying the process of data extraction, it allows investigators to focus on downstream analysis and to facilitate reproducible science. 

### More about PIC-SURE
The API is available in two different programming languages, python and R, enabling investigators to query the database the same way using either language.

PIC-SURE is a larger project from which the R and python PIC-SURE APIs are only a small part. Among other things, PIC-SURE also offers a graphical user interface that allows researchers to explore variables across multiple studies, filter participants that match criteria, and create cohorts from this interactive exploration.

The python API is actively developed by the Avillach Lab at Harvard Medical School.

PIC-SURE API GitHub repository:
* https://github.com/hms-dbmi/pic-sure-python-adapter-hpds

 ------- 

## Getting your user-specific security token

**Before running this notebook, please be sure to review the "Get your security token" documentation, which exists in the [`README.md` file](../README.md). It explains how to get a security token, which is mandatory to use the PIC-SURE API.**

To set up your token file, be sure to run the [`Workspace_setup.ipynb` file](./Workspace_setup.ipynb).

## Environment set-up

### Pre-requisites
* python 3.10 or later
* pip python package manager, already available in most systems with a python interpreter installed

### Install packages
The first step to using the PIC-SURE API is to install the `picsure` package. One package now replaces the three legacy packages (`PicSureClient`, `PicSureHpdsLib`, `PicSureBdcAdapter`).

**Note that if you are using the dedicated PIC-SURE environment within the *BDC Powered by Seven Bridges* platform, the necessary packages have already been installed.**

In [1]:
import sys

# Install/repair dependencies in the active kernel environment.
# Using sys.executable avoids mixing different Python/pip environments.
# Pinning a compatible pair avoids NumPy/Matplotlib binary ABI mismatches.
!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install --upgrade --force-reinstall "numpy<2" "matplotlib<3.9"

# Install picsure package.
!{sys.executable} -m pip install --upgrade --force-reinstall git+https://github.com/hms-dbmi/pic-sure-python-adapter-hpds.git@main

import pandas as pd
import matplotlib.pyplot as plt
# BDC Powered by Terra users uncomment the following line to specify package install location
# sys.path.insert(0, r"/home/jupyter/.local/lib/python3.7/site-packages")

  Using cached numpy-1.26.4-cp311-cp311-macosx_11_0_arm64.whl.metadata (114 kB)
  Using cached matplotlib-3.8.4-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.8 kB)
  Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp311-cp311-macosx_10_9_universal2.whl.metadata (117 kB)
  Using cached kiwisolver-1.5.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.1 kB)
  Using cached packaging-26.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached pillow-12.2.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached numpy-1.26.4-cp311-cp311-macosx_11_0_arm64.whl (14.0 MB)
Using cached matplotlib-3.8.4-cp311-cp311-macosx_11_0_arm64.whl (7.5 MB)
Using cached c

In [2]:
import picsure
from picsure import Platform, ClauseType, GroupOperator, createClause, buildClauseGroup

## Connecting to a PIC-SURE resource

The following is required to get access to the PIC-SURE API:
* a platform (a named `Platform` enum, or a custom URL string)
* a user-specific security token

The code below connects to *BDC Authorized* using `Platform.BDC_PREDEV_AUTHORIZED` and the token saved in `token.txt` by `Workspace_setup.ipynb`.

If you have not already retrieved your user-specific token, please refer to the `Workspace_setup.ipynb` file.

In [3]:
token_file = "token.txt"
with open(token_file, "r") as f:
    my_token = f.read()

session = picsure.connect(platform=Platform.BDC_PREDEV_AUTHORIZED, token=my_token)

You're successfully connected to BDC Authorized as user george_colon@hms.harvard.edu!
Your token expires on unknown.


## Inspecting the session
In v2 each adapter object exposed a `help()` method. In v3 the `Session` returned by `picsure.connect()` exposes its methods directly — use `help(session)` or your IDE's autocomplete to discover them. The main ones you'll use below are `session.search`, `session.runQuery`, and `session.exportCSV`. A list of accessible resources is available via `session.getResourceID()` — shown below.

In [4]:
# `bdc.help()` equivalent: session exposes methods directly.
# To see available resources: session.getResourceID()
session.getResourceID()

,uuid,name,description
0,70c837be-5ffc-11eb-ae93-0242ac130002,open-hpds,
1,ac004461-1b47-4832-80e2-22a4aecabe39,open-hpds-v3,
2,ca0ad4a9-130a-3a8a-ae00-e35b07f1108b,visualization,
3,02e23f52-f354-4e8b-992c-d37c8b9ba140,auth-hpds,
4,36363664-6231-6134-2d38-6538652d3131,dictionary,


For example, the above output lists and briefly defines the four methods that can be used with the `bdc` resource. 

## Using the PIC-SURE variables dictionary
For the rest of this example notebook, we will use one of the publicly available datasets available on PIC-SURE. This dataset is the "Framingham Heart Study: Dataset for Teaching Purposes", which has a study accession of `tutorial-biolincc_framingham` in PIC-SURE. To find the variables related to this dataset, we will use `session.search()` with a term of interest — in this case, `tutorial`.

In [5]:
search_term = "tutorial"

In [6]:
my_variables = session.search(search_term)

In [7]:
len(my_variables)  # how many variables did the search return?

139

`session.search()` already returns a pandas DataFrame — no conversion step is needed. Preview the first rows:

In [8]:
my_variables_df = my_variables  # session.search() already returns a DataFrame
my_variables_df.head(5)  # view the first 5 rows

,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
0,\tutorial-biolincc_camp\AGEHOME\,AGEHOME,AGEHOME,,Categorical,tutorial-biolincc_camp,"[0, 1, 10, 100, 102, 103, 104, 105, 11, 110, 1...",NaN,NaN,True,None,biolincc_camp
1,\tutorial-biolincc_camp\AGE_RZ\,AGE_RZ,AGE_RZ,,Continuous,tutorial-biolincc_camp,[],5.0,13.0,True,None,biolincc_camp
2,\tutorial-biolincc_camp\ANYPET\,ANYPET,ANYPET,,Categorical,tutorial-biolincc_camp,"[No, Yes]",NaN,NaN,True,None,biolincc_camp
3,\tutorial-biolincc_camp\ANY_SMOKES\,ANY_SMOKES,ANY_SMOKES,,Categorical,tutorial-biolincc_camp,"[No, Yes]",NaN,NaN,True,None,biolincc_camp
4,\tutorial-biolincc_camp\DEHUMID\,DEHUMID,DEHUMID,,Categorical,tutorial-biolincc_camp,"[DK, No, Yes]",NaN,NaN,True,None,biolincc_camp


PIC-SURE integrates clinical and genomic datasets across *BDC*, including TOPMed and TOPMed-related studies, COVID-19 studies, and BioLINCC studies. Each variable is organized as a concept path that contains information about the study, variable group, and variable. Though the specifics of the concept paths are dependent on the type of study, the overall information included is the same.

Data Organization in PIC-SURE
---------------------------------------
| Data organization | TOPMed & TOPMed-related studies | BioLINCC & COVID-19 studies (including public data) |
|-------------------|---------------------------------|-----------------------------|
| General organization | Data organized using the format implemented by the database of Genotypes and Phenotypes (dbGaP). Generally, a given study will have several tables, and those tables have several variables. | Data do not follow dbGaP format; there are no phv or pht accessions. Data are organized in groups of like variables, when available. For example, variables like Age, Gender, and Race could be part of the Demographics variable group. |
| Concept path structure | \phs\pht\phv\variable name\ | \phs\variable name |
| Variable ID | phv corresponding to the variable accession number | Equivalent to variable name |
| Variable name | Encoded variable name that was used by the original submitters of the data | Encoded variable name that was used by the original submitters of the data |
| Variable description | Description of the variable | Description of the variable, as available |
| Dataset ID | pht corresponding to the trait table accession number | Equivalent to Dataset name |
| Dataset name | Name of the trait table | Name of a group of like variables, as available |
| Dataset description | Description of the trait table | Description of a group of like variables, as available |
| Study ID | phs corresponding to the study accession number | phs corresponding to the study accession number |
| Study description | Description of the study from dbGaP | Description of the study from dbGaP |

*Note: in the v3 search DataFrame the concept path is exposed in the `conceptPath` column. This is what you pass to `createClause()` when building queries.*

The `conceptPath` column of the search DataFrame holds each variable's concept path — this is the value you pass to `createClause()` when building queries. Below we preview the first ten paths:

In [9]:
my_variables_df["conceptPath"].iloc[:10].tolist()  # first 10 concept paths

['\\tutorial-biolincc_camp\\AGEHOME\\',
 '\\tutorial-biolincc_camp\\AGE_RZ\\',
 '\\tutorial-biolincc_camp\\ANYPET\\',
 '\\tutorial-biolincc_camp\\ANY_SMOKES\\',
 '\\tutorial-biolincc_camp\\DEHUMID\\',
 '\\tutorial-biolincc_camp\\ETHNIC\\',
 '\\tutorial-biolincc_camp\\FDAYS\\',
 '\\tutorial-biolincc_camp\\GENDER\\',
 '\\tutorial-biolincc_camp\\HEMOG\\',
 '\\tutorial-biolincc_camp\\ID\\']

To view additional information for an individual variable, index into the search DataFrame directly — each row carries the concept path, name, description, data type, and (for categorical variables) the list of values.

In [10]:
first_var = my_variables_df["conceptPath"].iloc[0]
my_variables_df.iloc[0]  # variable details (replaces varInfo)

conceptPath                        \tutorial-biolincc_camp\AGEHOME\
name                                                        AGEHOME
display                                                     AGEHOME
description                                                        
dataType                                                Categorical
studyId                                      tutorial-biolincc_camp
values            [0, 1, 10, 100, 102, 103, 104, 105, 11, 110, 1...
min                                                             NaN
max                                                             NaN
allowFiltering                                                 True
meta                                                           None
studyAcronym                                          biolincc_camp
Name: 0, dtype: object

Now you can try to search for a term on your own. Below is sample code on how to search for the term `sex`. To practice searching the data dictionary, you can change "sex" to a term you are interested in. You will see the results displayed in the convenient dataframe format using the `displayResults()` method. Note - the results displayed will show results from all studies you have access to. 

In [11]:
my_search = session.search("sex")
len(my_search)
# my_search.head()  # show matching variables

2596

## Using PIC-SURE to build a query and retrieve data
You can also use the PIC-SURE API to build a query and retrieve data. With this functionality, you can filter based on specific variables, add others, and export the data as a DataFrame into this notebook.

In v3 a query is not a stateful object you mutate; it is a **clause tree** you construct with `createClause()` and combine with `buildClauseGroup()`. Each clause has a **type** and targets one concept path; groups combine clauses with AND or OR.

The four clause types (`ClauseType` enum) are:

| Clause type | Purpose | Produces in results |
|---|---|---|
| `ClauseType.SELECT` | Include a variable in the output without filtering | Variable column, no record subsetting |
| `ClauseType.REQUIRE` | Include a variable and require non-null values | Variable column, only records where it has a value |
| `ClauseType.ANYRECORD` | Include only records that have at least one non-null value for the variable | Variable column, only records with at least one value |
| `ClauseType.FILTER` | Restrict to records matching a value or range (`categories=...`, `min=...`, `max=...`) | Variable column, only records matching the filter |

Combine one or more clauses into a query with `buildClauseGroup([...], root=GroupOperator.AND)` (or `GroupOperator.OR`).

As an example query, let's use the Framingham tutorial dataset to investigate the prevalence of hypertension and distribution of age of current smokers with body mass index greater than 20.

In [12]:
# Ensure that only Framingham tutorial variables are shown in the data dictionary, which can vary based on individual access
phs_number = "tutorial-biolincc_framingham"
tutorial_df = my_variables_df[my_variables_df.studyId == phs_number]

### Build a query with a categorical variable - Current smoker
Let's practice building a query by filtering on variables. Based on the search for the Framingham tutorial dataset variables, we can save the concept path of the "Current cigarette smoking at exam" variable, which is a categorical variable. 

In [13]:
smoke_variable_path = tutorial_df.conceptPath[
    tutorial_df.description == "Current cigarette smoking at exam"
].item()
smoke_variable_path

'\\tutorial-biolincc_framingham\\CURSMOKE\\'

We can take a look at the options of values to filter by using the `values` column of the data dictionary.

In [14]:
tutorial_df["values"][tutorial_df.description == "Current cigarette smoking at exam"]

101    [Current smoker, Not current smoker]
Name: values, dtype: object

Let's apply a filter on the "Current cigarette smoking at exam" variable to only select participants with "Current smoker." Note that though we are only filtering by one value, you can pass a list of values to the `categories=` argument to match any of them.

In [15]:
smoke_clause = createClause(
    smoke_variable_path,
    type=ClauseType.FILTER,
    categories="Current smoker",
)

### Build a query with a continuous variable - BMI

Let's practice building a query by filtering on a continuous variable, in this case, BMI. We can find the BMI concept path using a similar approach as above.

In [16]:
bmi_variable_path = tutorial_df.conceptPath[
    tutorial_df.name == "BMI"
].item()
bmi_variable_path

'\\tutorial-biolincc_framingham\\BMI\\'

In v2 the dictionary exposed `min` and `max` columns for continuous variables. The new `session.search()` DataFrame does not return those; the cell below shows what the new DataFrame carries for BMI.

In [17]:
# Note: the new session.search() DataFrame does not include min/max
# columns. For categorical variables, inspect the `values` column;
# for continuous variables like BMI, range information is no longer
# returned by the dictionary API.
tutorial_df[tutorial_df["name"] == "BMI"][["conceptPath", "dataType", "description"]]

,conceptPath,dataType,description
98,\tutorial-biolincc_framingham\BMI\,Continuous,"Body Mass Index, weight in kilograms/height me..."


Let's apply a filter on the "Body Mass Index, weight in kilograms/height meters squared" variable to select only participants with values greater than 20. Note that while in this example only a `min` is specified, a `max` can also be defined for the filter.

In [18]:
bmi_clause = createClause(bmi_variable_path, type=ClauseType.FILTER, min=20)

### Adding variables to include in export - Age and Hypertension
In addition to adding filters, specific variables can be included in the export for analysis. Let's do this for the "Age at exam (years)" and "Hypertensive. Defined as the first exam treated for high blood pressure or second exam in which either Systolic is 6 140 mmHg or Diastolic 6 90mmHg" variables.

In [19]:
age_variable_path = tutorial_df.loc[tutorial_df.description == "Age at exam (years)", "conceptPath"].item()
hyperten_variable_path = tutorial_df.loc[tutorial_df.name == "HYPERTEN", "conceptPath"].item()

Let's add these variables to our query. The clause type determines which records are kept:

* `ClauseType.SELECT` — adds the variable to the output for all records passing the upstream filters, regardless of whether they have a value.
* `ClauseType.ANYRECORD` — adds the variable and keeps only records that have at least one non-null value for it.
* `ClauseType.REQUIRE` — adds the variable and keeps only records that have a non-null value.

For this, let's use `ClauseType.REQUIRE` to only select participants that have information for both of these variables.

In [20]:
age_clause = createClause(age_variable_path, type=ClauseType.REQUIRE)
hyperten_clause = createClause(hyperten_variable_path, type=ClauseType.REQUIRE)

query_example = buildClauseGroup(
    [smoke_clause, bmi_clause, age_clause, hyperten_clause],
    root=GroupOperator.AND,
)

### Exporting participant-level data from the query
The query has been constructed and can now be exported for analysis. 

In the data dictionary dataframe shown previously, each row represented a single concept path or variable. In the query dataframe, the concept paths are added as columns with each row representing a participant with data that matches your query. 

The dataframe above should contain some automatically exported concept paths, such as `patient_id`, `Parent Study Accession with Subject ID`, `Topmed Study Accession with Subject ID`, and `consents`, and the concept paths we added to our query.

In [21]:
example_results = session.runQuery(query_example, type="participant")
example_results.head()

,patient_id
0,192514
1,192513
2,192523
3,192526
4,192524


As you can see in the results, there are some instances where study participants may have more than one value for a given variable. For example, this may be the case when a study participants answers questionnaires for multiple visits. 

In the PIC-SURE output, this is shown as values being separated by a tab or `\t` value. These multiple values will need to be accounted for depending on the planned analysis.

With this example, averages of the age and BMI values could be calculated and a new variable "ever smoker" could be created based on whether "current smoker" was ever answered for the CURSMOKE variable. The code below shows this example of how to handle these values.

*Note: Approaches to handling multiple values will differ based on the approach.*

In [22]:
# Select rows of interest and rename them
clean_results = example_results[["\\tutorial-biolincc_framingham\\AGE\\", "\\tutorial-biolincc_framingham\\BMI\\", "\\tutorial-biolincc_framingham\\CURSMOKE\\", "\\tutorial-biolincc_framingham\\HYPERTEN\\"]]
clean_results.columns = ["AGE", "BMI", "CURSMOKE", "HYPERTEN"]

KeyError: "None of [Index(['\\tutorial-biolincc_framingham\\AGE\\',\n       '\\tutorial-biolincc_framingham\\BMI\\',\n       '\\tutorial-biolincc_framingham\\CURSMOKE\\',\n       '\\tutorial-biolincc_framingham\\HYPERTEN\\'],\n      dtype='object')] are in the [columns]"

In [ ]:
# Function that splits the values and calculates the mean
import statistics
def mean_multiple_values(df_values):
    sep_values = str(df_values).split("\t")
    mean_val = statistics.mean([float(i) for i in sep_values])
    return(mean_val)

# Apply the function to calculate means to the AGE and BMI variables
clean_results.loc[:, "mean_age"] = clean_results.loc[:, "AGE"].apply(mean_multiple_values)
clean_results.loc[:, "mean_bmi"] = clean_results.loc[:, "BMI"].apply(mean_multiple_values)

In [ ]:
# Function that flags participants as smoker if they have an answer of "Current smoker"
def ever_smoker(smoke_vals):
    sep_smoke_vals = smoke_vals.split("\t")
    if "Current smoker" in sep_smoke_vals:
        return("Smoker")
    else:
        return("Non-smoker")
    
# Apply the function to identify smokers to the CURSMOKE variable
clean_results.loc[:, "ever_smoker"] = clean_results.loc[:, "CURSMOKE"].apply(ever_smoker)

In [ ]:
# Take a look at the new results
clean_results[["mean_age", "mean_bmi", "ever_smoker","HYPERTEN"]]